<a href="https://colab.research.google.com/github/AyoubElKharraf/NeRF_LRM__From_2D_to_3D/blob/main/01_env_check_nerf.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import sys
import torch
import subprocess

print("=== 1. INFORMATIONS SYSTÈME & PYTHON ===")
print(f"Version Python : {sys.version.split()[0]}")
print(f"Version PyTorch : {torch.__version__}")

print("\n=== 2. VÉRIFICATION CUDA & GPU ===")
cuda_available = torch.cuda.is_available()
print(f"CUDA disponible : {cuda_available}")

if cuda_available:
    device_count = torch.cuda.device_count()
    current_device = torch.cuda.current_device()
    gpu_name = torch.cuda.get_device_name(current_device)
    vram_total = torch.cuda.get_device_properties(current_device).total_memory / (1024**3)

    print(f"Nombre de GPU : {device_count}")
    print(f"Modèle de GPU : {gpu_name}")
    print(f"VRAM Totale : {vram_total:.2f} Go")
    print(f"Version CUDA compilée avec PyTorch : {torch.version.cuda}")
    print(f"Version cuDNN : {torch.backends.cudnn.version()}")

    # Test de calcul réel sur le GPU (Allocation de tenseur)
    try:
        x = torch.rand(1000, 1000, device="cuda")
        y = torch.matmul(x, x)
        print("Test de calcul matriciel GPU : SUCCÈS ✅")
    except Exception as e:
        print(f"Test de calcul matriciel GPU : ÉCHEC ❌ ({e})")
else:
    print("ATTENTION : Aucun GPU détecté. Vérifie 'Modifier le type d'exécution' -> GPU.")

print("\n=== 3. SORTIE MATÉRIELLE NVIDIA-SMI ===")
try:
    smi_output = subprocess.check_output(["nvidia-smi"]).decode("utf-8")
    print(smi_output)
except Exception as e:
    print(f"Impossible d'exécuter nvidia-smi : {e}")

=== 1. INFORMATIONS SYSTÈME & PYTHON ===
Version Python : 3.13.16
Version PyTorch : 2.14.1+cu130

=== 2. VÉRIFICATION CUDA & GPU ===
CUDA disponible : True
Nombre de GPU : 1
Modèle de GPU : Tesla T4
VRAM Totale : 14.56 Go
Version CUDA compilée avec PyTorch : 13.0
Version cuDNN : 92800
Test de calcul matriciel GPU : SUCCÈS ✅

=== 3. SORTIE MATÉRIELLE NVIDIA-SMI ===
Mon Nov 16 15:17:54 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+======

In [1]:
!pip install nerfstudio

In [2]:
!ns-train --help

/usr/local/lib/python3.13/dist-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
I0000 00:00:1794843208.679189    5549 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/usr/local/lib/python3.13/dist-packages/nerfstudio/field_components/activations.py:32: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @custom_fwd(cast_inputs=torch.float32)
/usr/local/lib/python3.13/dist-packages/nerfstudio/field_components/activations.py:38: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. Please use `torch.amp.custom_bwd(args..., device_type='cuda')` instead.
  @custom_bwd
usage

In [5]:
!pip install -q huggingface_hub

In [6]:
from huggingface_hub import hf_hub_download
import zipfile
from pathlib import Path

archive = hf_hub_download(
    repo_id="nerfbaselines/nerfbaselines-data",
    repo_type="dataset",
    filename="blender/lego.zip",
    local_dir="/content/data"
)

output_dir = Path("/content/data/blender")
output_dir.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(archive) as zip_file:
    zip_file.extractall(output_dir)

print("Dataset téléchargé dans :", output_dir)

blender/lego.zip: reconstructing file:   0%|          |  0.00B /  180MB            

blender/lego.zip: downloading bytes:           |  0.00B            

Dataset téléchargé dans : /content/data/blender


In [7]:
!find /content/data/blender -name "transforms*.json"

/content/data/blender/lego/transforms_val.json
/content/data/blender/lego/transforms_test.json
/content/data/blender/lego/transforms_train.json


In [10]:
!ns-train nerfacto \
    --max-num-iterations 1000 \
    blender-data \
    --data /content/data/blender/lego

/usr/local/lib/python3.13/dist-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
I0000 00:00:1794844307.306800   10147 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/usr/local/lib/python3.13/dist-packages/nerfstudio/field_components/activations.py:32: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @custom_fwd(cast_inputs=torch.float32)
/usr/local/lib/python3.13/dist-packages/nerfstudio/field_components/activations.py:38: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. Please use `torch.amp.custom_bwd(args..., device_type='cuda')` instead.
  @custom_bwd
─────

In [9]:
!find /content/data -type d -name "lego"

/content/data/blender/lego


In [11]:
!nvidia-smi

Mon Nov 16 16:53:03 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   74C    P0             34W /   70W |       0MiB /  15360MiB |     51%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [12]:
!find /content/outputs -type f | head -50

/content/outputs/unnamed/nerfacto/2026-11-16_155153/config.yml
/content/outputs/unnamed/nerfacto/2026-11-16_155153/nerfstudio_models/step-000000999.ckpt
/content/outputs/unnamed/nerfacto/2026-11-16_155153/dataparser_transforms.json


In [13]:
!find /content/outputs -name "*.ckpt"

/content/outputs/unnamed/nerfacto/2026-11-16_155153/nerfstudio_models/step-000000999.ckpt
